# **Density Assignment**

## **Combination: Trees + Wood Density Databases**

In [14]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from openpyxl import load_workbook


# ============================================================
# PFADE
# ============================================================

INVENTORY = Path(
    r"Z:\Ghana\RCT_outputs\BO1_preseg_RCT\output\rct\segmented"
    r"\BO1_Trees_Inventory\BO1_tree_inventory_master.xlsx"
)

GWDD_SPECIES = Path(
    r"C:\Users\laudenb\Documents\Density_Files\World_Density_V2"
    r"\gwddagg_v2.2_binomial.csv"
)

GWDD_GENUS = Path(
    r"C:\Users\laudenb\Documents\Density_Files\World_Density_V2"
    r"\gwddagg_v2.2_genus.csv"
)

ICRAF = Path(
    r"C:\Users\laudenb\Documents\Density_Files"
    r"\African wood density database.xlsx"
)

NTONMEN = Path(
    r"C:\Users\laudenb\Documents\Density_Files"
    r"\Ntonmen_Appendix1_Wood_Density.xlsx"
)

OUTPUT = INVENTORY.with_name(
    INVENTORY.stem + "_with_wood_density.xlsx"
)


# ============================================================
# BEKANNTE SCHREIBFEHLER
# ============================================================

SPECIES_CORRECTIONS = {
    "minilkara multinavis": "Manilkara multinervis",
    "prosopsis africana": "Prosopis africana",
    "hyminocardia acida": "Hymenocardia acida",
    "magaritaria discoidea": "Margaritaria discoidea",
}


# ============================================================
# HILFSFUNKTIONEN
# ============================================================

def clean_name(value):
    if value is None or pd.isna(value):
        return ""

    name = re.sub(r"\s+", " ", str(value).strip())
    return SPECIES_CORRECTIONS.get(name.casefold(), name)


def key(value):
    return clean_name(value).casefold()


def genus(value):
    name = clean_name(value)
    return name.split()[0] if name else ""


def to_number(series):
    return pd.to_numeric(
        series.astype(str)
        .str.strip()
        .str.replace(",", ".", regex=False)
        .replace({"": np.nan, "/": np.nan, "nan": np.nan}),
        errors="coerce",
    )


def make_lookup(df, name_col, value_col):
    temp = pd.DataFrame({
        "name": df[name_col].map(key),
        "value": to_number(df[value_col]),
    })

    temp = temp[
        (temp["name"] != "") &
        temp["value"].notna()
    ]

    return temp.groupby("name")["value"].mean().to_dict()


def mean_available(*values):
    values = [
        float(v) for v in values
        if v is not None and pd.notna(v)
    ]
    return float(np.mean(values)) if values else None


# ============================================================
# 1. GWDD
# ============================================================

gwdd_species_df = pd.read_csv(GWDD_SPECIES, sep=",")
gwdd_genus_df = pd.read_csv(GWDD_GENUS, sep=",")

gwdd_species = make_lookup(
    gwdd_species_df, "binomial", "wsg_est"
)

gwdd_genus = make_lookup(
    gwdd_genus_df, "genus", "wsg_est"
)


# ============================================================
# 2. ICRAF
# ============================================================

# Excel-Zeile 6 enthält die Überschriften
icraf = pd.read_excel(
    ICRAF,
    sheet_name="Wood density 2024",
    header=5,
)

# Air-dry density -> Basic Wood Density
icraf["basic_density"] = (
    to_number(icraf["Wood density average"]) * 0.82
)

# Aktuelle und alte Artnamen als mögliche Treffer
icraf_current = icraf[
    ["scientificName", "basic_density"]
].rename(columns={"scientificName": "name"})

icraf_old = icraf[
    ["Species Name 2012", "basic_density"]
].rename(columns={"Species Name 2012": "name"})

icraf_names = pd.concat(
    [icraf_current, icraf_old],
    ignore_index=True,
)

icraf_species = make_lookup(
    icraf_names, "name", "basic_density"
)

# Gattungsmittel nur anhand der aktuellen Taxonomie
icraf["genus"] = icraf["scientificName"].map(genus)

icraf_genus = make_lookup(
    icraf, "genus", "basic_density"
)


# ============================================================
# 3. NTONMEN
# ============================================================

ntonmen = pd.read_excel(
    NTONMEN,
    sheet_name="Appendix 1",
)

ntonmen_species = make_lookup(
    ntonmen,
    "Species",
    "Measure wood density - average",
)

ntonmen["genus"] = ntonmen["Species"].map(genus)

ntonmen_genus = make_lookup(
    ntonmen,
    "genus",
    "Measure wood density - average",
)


# ============================================================
# ZUORDNUNGSLOGIK
# ============================================================

def assign_density(original_species):
    corrected = clean_name(original_species)
    species_key = key(corrected)
    genus_key = key(genus(corrected))

    result = {
        "Species_Corrected": corrected,
        "Wood_Density": None,
        "WD_Source": None,
        "WD_Level": "missing",
        "WD_GWDD": None,
        "WD_ICRAF": None,
        "WD_Ntonmen": None,
        "WD_Status": "NO_GENUS_VALUE",
    }

    if not species_key:
        result["WD_Status"] = "NO_SPECIES_NAME"
        return result

    # --------------------------------------------------------
    # Artwerte: zuerst GWDD und ICRAF
    # --------------------------------------------------------

    gwdd_value = gwdd_species.get(species_key)
    icraf_value = icraf_species.get(species_key)

    result["WD_GWDD"] = gwdd_value
    result["WD_ICRAF"] = icraf_value

    main_species_mean = mean_available(
        gwdd_value,
        icraf_value,
    )

    if main_species_mean is not None:
        result["Wood_Density"] = main_species_mean
        result["WD_Level"] = "species"

        if gwdd_value is not None and icraf_value is not None:
            result["WD_Source"] = "GWDD + ICRAF"
            result["WD_Status"] = "SPECIES_MEAN"
        elif gwdd_value is not None:
            result["WD_Source"] = "GWDD"
            result["WD_Status"] = "SPECIES_GWDD"
        else:
            result["WD_Source"] = "ICRAF"
            result["WD_Status"] = "SPECIES_ICRAF"

        return result

    # --------------------------------------------------------
    # Ntonmen als Art-Backup
    # --------------------------------------------------------

    ntonmen_value = ntonmen_species.get(species_key)
    result["WD_Ntonmen"] = ntonmen_value

    if ntonmen_value is not None:
        result["Wood_Density"] = ntonmen_value
        result["WD_Source"] = "Ntonmen"
        result["WD_Level"] = "species"
        result["WD_Status"] = "SPECIES_NTONMEN_BACKUP"
        return result

    # --------------------------------------------------------
    # Gattungsebene
    # --------------------------------------------------------

    if not genus_key:
        result["WD_Status"] = "NO_GENUS_NAME"
        return result

    gwdd_genus_value = gwdd_genus.get(genus_key)
    icraf_genus_value = icraf_genus.get(genus_key)

    result["WD_GWDD"] = gwdd_genus_value
    result["WD_ICRAF"] = icraf_genus_value

    main_genus_mean = mean_available(
        gwdd_genus_value,
        icraf_genus_value,
    )

    if main_genus_mean is not None:
        result["Wood_Density"] = main_genus_mean
        result["WD_Level"] = "genus"

        if (
            gwdd_genus_value is not None
            and icraf_genus_value is not None
        ):
            result["WD_Source"] = "GWDD + ICRAF"
            result["WD_Status"] = "GENUS_MEAN"
        elif gwdd_genus_value is not None:
            result["WD_Source"] = "GWDD"
            result["WD_Status"] = "GENUS_GWDD"
        else:
            result["WD_Source"] = "ICRAF"
            result["WD_Status"] = "GENUS_ICRAF"

        return result

    # --------------------------------------------------------
    # Ntonmen als Gattungs-Backup
    # --------------------------------------------------------

    ntonmen_genus_value = ntonmen_genus.get(genus_key)
    result["WD_Ntonmen"] = ntonmen_genus_value

    if ntonmen_genus_value is not None:
        result["Wood_Density"] = ntonmen_genus_value
        result["WD_Source"] = "Ntonmen"
        result["WD_Level"] = "genus"
        result["WD_Status"] = "GENUS_NTONMEN_BACKUP"

    return result


# ============================================================
# INVENTAR ÖFFNEN
# ============================================================

workbook = load_workbook(INVENTORY)

worksheet = None
header_row = None
species_col = None

# Sucht die Spalte Species in allen Blättern
for ws in workbook.worksheets:
    for row in range(1, min(ws.max_row, 20) + 1):
        headers = {
            str(cell.value).strip(): cell.column
            for cell in ws[row]
            if cell.value is not None
        }

        if "Species" in headers:
            worksheet = ws
            header_row = row
            species_col = headers["Species"]
            break

    if worksheet is not None:
        break

if worksheet is None:
    raise ValueError("Die Spalte 'Species' wurde nicht gefunden.")


# ============================================================
# AUSGABESPALTEN
# ============================================================

output_columns = [
    "Species_Corrected",
    "Wood_Density",
    "WD_Source",
    "WD_Level",
    "WD_GWDD",
    "WD_ICRAF",
    "WD_Ntonmen",
    "WD_Status",
]

existing_headers = {
    str(cell.value).strip(): cell.column
    for cell in worksheet[header_row]
    if cell.value is not None
}

column_numbers = {}

for name in output_columns:
    if name in existing_headers:
        column_numbers[name] = existing_headers[name]
    else:
        new_col = worksheet.max_column + 1
        worksheet.cell(header_row, new_col, name)
        column_numbers[name] = new_col


# ============================================================
# ZEILEN VERARBEITEN
# ============================================================

cache = {}
missing = []

for row in range(header_row + 1, worksheet.max_row + 1):
    original_species = worksheet.cell(
        row,
        species_col,
    ).value

    cache_key = key(original_species)

    if cache_key not in cache:
        cache[cache_key] = assign_density(original_species)

    result = cache[cache_key]

    for name, value in result.items():
        cell = worksheet.cell(
            row,
            column_numbers[name],
            value,
        )

        if name in {
            "Wood_Density",
            "WD_GWDD",
            "WD_ICRAF",
            "WD_Ntonmen",
        }:
            cell.number_format = "0.000"

    if result["Wood_Density"] is None:
        missing.append([
            row,
            original_species,
            result["Species_Corrected"],
            genus(result["Species_Corrected"]),
            result["WD_Status"],
        ])


# ============================================================
# BLATT MIT FEHLENDEN ARTEN
# ============================================================

if "WD_Missing" in workbook.sheetnames:
    del workbook["WD_Missing"]

missing_sheet = workbook.create_sheet("WD_Missing")

missing_sheet.append([
    "Excel_Row",
    "Species_Original",
    "Species_Corrected",
    "Genus",
    "WD_Status",
])

for record in missing:
    missing_sheet.append(record)

missing_sheet.freeze_panes = "A2"


# ============================================================
# SPEICHERN
# ============================================================

workbook.save(OUTPUT)

print("Fertig.")
print("Ausgabedatei:")
print(OUTPUT)
print()
print("Nicht zugeordnete Zeilen:", len(missing))

FileNotFoundError: [Errno 2] No such file or directory: 'Z:\\Ghana\\RCT_outputs\\BO1_preseg_RCT\\output\\rct\\segmented\\BO1_Trees_Inventory\\BO1_tree_inventory_master.xlsx'

Dieser Codeblock liest die Inventardaten sowie die drei Holzdichtequellen GWDD, ICRAF und Ntonmen ein. Bekannte Schreibfehler in den Artnamen werden korrigiert, anschließend wird für jeden Baum zunächst ein artspezifischer Holzdichtewert gesucht. Falls kein Artwert verfügbar ist, wird auf Gattungsebene gesucht; nicht zuordenbare Fälle werden entsprechend markiert. Die Ergebnisse werden zusammen mit Quelle, Zuordnungsebene und Kontrollinformationen in einer neuen Excel-Datei gespeichert.

## **Control Statistics**

In [13]:
# ============================================================
# STATISTIK DER HOLZDICHTE-ZUORDNUNG
# direkt nach dem großen Code ausführen
# ============================================================

records = []

for row in range(header_row + 1, worksheet.max_row + 1):

    species_original = worksheet.cell(
        row=row,
        column=species_col
    ).value

    record = {
        "Species": species_original
    }

    for name in output_columns:
        record[name] = worksheet.cell(
            row=row,
            column=column_numbers[name]
        ).value

    records.append(record)


stats = pd.DataFrame(records)


# ============================================================
# BÄUME
# ============================================================

total = len(stats)
assigned = stats["Wood_Density"].notna().sum()
missing_count = stats["Wood_Density"].isna().sum()

print("GESAMT")
print("------")
print("Bäume gesamt:", total)
print("Mit Holzdichte:", assigned)
print("Ohne Holzdichte:", missing_count)
print(f"Zugeordnet: {assigned / total * 100:.2f} %")
print(f"Nicht zugeordnet: {missing_count / total * 100:.2f} %")


# ============================================================
# STATUS
# ============================================================

print("\nZUORDNUNGSSTATUS")
print("----------------")
print(stats["WD_Status"].value_counts(dropna=False))


# ============================================================
# QUELLEN
# ============================================================

print("\nQUELLEN")
print("-------")
print(stats["WD_Source"].fillna("KEINE QUELLE").value_counts())


# ============================================================
# ART- / GATTUNGSEBENE
# ============================================================

print("\nZUORDNUNGSEBENE")
print("----------------")
print(stats["WD_Level"].value_counts(dropna=False))


# ============================================================
# EINDEUTIGE ARTEN
# ============================================================

species_stats = (
    stats.groupby(
        ["Species", "Species_Corrected"],
        dropna=False
    )
    .agg(
        Anzahl_Baeume=("Species", "size"),
        Wood_Density=("Wood_Density", "first"),
        WD_Source=("WD_Source", "first"),
        WD_Level=("WD_Level", "first"),
        WD_Status=("WD_Status", "first"),
    )
    .reset_index()
)

print("\nEINDEUTIGE ARTEN")
print("----------------")
print("Arten gesamt:", len(species_stats))
print(
    "Arten mit Holzdichte:",
    species_stats["Wood_Density"].notna().sum()
)
print(
    "Arten ohne Holzdichte:",
    species_stats["Wood_Density"].isna().sum()
)


# ============================================================
# NICHT ZUGEORDNET
# ============================================================

missing_species = (
    species_stats[
        species_stats["Wood_Density"].isna()
    ]
    .sort_values("Anzahl_Baeume", ascending=False)
)

print("\nNICHT ZUGEORDNETE ARTEN")
print("-----------------------")

if missing_species.empty:
    print("Alle Arten wurden zugeordnet.")
else:
    display(missing_species)

GESAMT
------
Bäume gesamt: 95
Mit Holzdichte: 95
Ohne Holzdichte: 0
Zugeordnet: 100.00 %
Nicht zugeordnet: 0.00 %

ZUORDNUNGSSTATUS
----------------
WD_Status
SPECIES_GWDD     28
SPECIES_MEAN     25
GENUS_MEAN       23
SPECIES_ICRAF    18
GENUS_GWDD        1
Name: count, dtype: int64

QUELLEN
-------
WD_Source
GWDD + ICRAF    48
GWDD            29
ICRAF           18
Name: count, dtype: int64

ZUORDNUNGSEBENE
----------------
WD_Level
species    71
genus      24
Name: count, dtype: int64

EINDEUTIGE ARTEN
----------------
Arten gesamt: 22
Arten mit Holzdichte: 22
Arten ohne Holzdichte: 0

NICHT ZUGEORDNETE ARTEN
-----------------------
Alle Arten wurden zugeordnet.


Dieser Codeblock wertet die Ergebnisse der Holzdichtezuordnung aus. Er zeigt, wie viele Bäume und Arten erfolgreich zugeordnet wurden, welche Quellen verwendet wurden und wie viele Werte auf Art- bzw. Gattungsebene beruhen. Zusätzlich werden alle Arten aufgelistet, für die keine Holzdichte gefunden werden konnte.